# 📊 Análise de Dados COVID-19 — Espírito Santo
## Metodologia KDD (Knowledge Discovery in Databases)

**Objetivo:** Explorar, limpar e visualizar os dados de notificações de COVID-19 do ES.

**Dataset:** `MICRODADOS.csv` — separador `;`, encoding `latin-1`

---
## 🔧 Célula Inicial — Importações e Leitura do Dataset

In [ ]:
# Importação das bibliotecas necessárias
import pandas as pd                        # Manipulação de dados
import matplotlib.pyplot as plt            # Geração de gráficos
import matplotlib.ticker as mticker        # Formatação de eixos
import warnings
warnings.filterwarnings('ignore')          # Suprime avisos desnecessários

# Configuração global dos gráficos
plt.rcParams['figure.figsize'] = (12, 6)  # Tamanho padrão dos gráficos
plt.rcParams['font.size'] = 12            # Tamanho padrão da fonte

# Leitura do dataset
# sep=';'     -> separador de colunas é ponto-e-vírgula
# encoding='latin-1' -> encoding usado em arquivos brasileiros
# low_memory=False   -> evita avisos com colunas de tipos mistos
df = pd.read_csv('MICRODADOS.csv', sep=';', encoding='latin-1', low_memory=False)

print(f'✅ Dataset carregado com sucesso!')
print(f'   Linhas: {df.shape[0]:,}  |  Colunas: {df.shape[1]}')

---
## Exercício 1 — Visão Geral do Dataset

**Objetivo:** Entender a estrutura do dataset antes de qualquer análise.
Isso é fundamental na etapa de **Seleção e Pré-processamento** do KDD.

In [ ]:
# --- 1.1 Dimensões do dataset ---
linhas, colunas = df.shape
print(f'📐 Dimensões: {linhas:,} linhas × {colunas} colunas\n')

# --- 1.2 Primeiras linhas ---
print('📋 Primeiras 5 linhas:')
display(df.head())

# --- 1.3 Tipos de dados ---
print('\n🔠 Tipos de dados por coluna:')
print(df.dtypes)

# --- 1.4 Valores nulos ---
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df) * 100).round(2)
resumo_nulos = pd.DataFrame({'Nulos': nulos, '% Nulos': nulos_pct})
resumo_nulos = resumo_nulos[resumo_nulos['Nulos'] > 0].sort_values('Nulos', ascending=False)

print(f'\n🔴 Colunas com valores nulos ({len(resumo_nulos)} de {colunas}):')
display(resumo_nulos)

### 📝 Interpretação — Exercício 1

O dataset contém milhares de registros de notificações de COVID-19 no Espírito Santo.
Observamos que:
- Várias colunas possuem valores ausentes (NaN), o que é comum em dados de saúde pública;
- Os tipos de dados incluem texto (object) e numérico — precisaremos converter datas;
- Esta etapa é essencial para planejar a limpeza e transformação dos dados (KDD).

---
## Exercício 2 — Distribuição por Classificação

**Objetivo:** Verificar como os casos estão distribuídos entre as categorias de classificação.

In [ ]:
# Conta a frequência de cada classificação
contagem = df['Classificacao'].value_counts()

# Calcula a porcentagem de cada categoria
porcentagem = (contagem / contagem.sum() * 100).round(2)

# Exibe tabela com contagem e porcentagem
tabela_class = pd.DataFrame({'Quantidade': contagem, 'Porcentagem (%)': porcentagem})
display(tabela_class)

# --- Gráfico de barras horizontal ---
fig, ax = plt.subplots(figsize=(12, 5))

cores = ['#e74c3c', '#3498db', '#2ecc71', '#f39c12', '#9b59b6']
barras = ax.barh(contagem.index, contagem.values,
                 color=cores[:len(contagem)], edgecolor='white')

# Adicionar rótulos nas barras
for barra, valor, pct in zip(barras, contagem.values, porcentagem.values):
    ax.text(barra.get_width() + contagem.max() * 0.01,
            barra.get_y() + barra.get_height() / 2,
            f'{valor:,} ({pct}%)', va='center', fontsize=11)

ax.set_title('📊 Distribuição dos Casos por Classificação', fontsize=14, fontweight='bold')
ax.set_xlabel('Número de Notificações')
ax.set_ylabel('Classificação')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 📝 Interpretação — Exercício 2

A distribuição por classificação revela o panorama geral das notificações:
- A maioria dos registros é de casos **Confirmados**, refletindo a abrangência do sistema de notificação;
- Casos **Descartados** e **Suspeitos** indicam a dinâmica de triagem;
- O gráfico horizontal facilita a comparação visual entre as categorias.

---
## Exercício 3 — Top 10 Municípios com Mais Notificações

**Objetivo:** Identificar os municípios com maior volume de notificações.

In [ ]:
# Seleciona os 10 municípios com mais registros
top10 = df['Municipio'].value_counts().head(10)

# --- Gráfico de barras vertical ---
fig, ax = plt.subplots(figsize=(13, 6))

# Gera lista de cores: líder em destaque (vermelho), demais em azul
cores = ['#e74c3c'] + ['#3498db'] * 9

barras = ax.bar(top10.index, top10.values, color=cores, edgecolor='white', width=0.7)

# Rótulo com valor em cima de cada barra
for barra in barras:
    ax.text(barra.get_x() + barra.get_width() / 2,
            barra.get_height() + top10.max() * 0.01,
            f'{int(barra.get_height()):,}',
            ha='center', fontsize=10, fontweight='bold')

ax.set_title('🏙️ Top 10 Municípios com Mais Notificações de COVID-19', fontsize=14, fontweight='bold')
ax.set_xlabel('Município')
ax.set_ylabel('Número de Notificações')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=30, ha='right')
ax.legend(
    handles=[plt.Rectangle((0,0),1,1, color='#e74c3c'),
             plt.Rectangle((0,0),1,1, color='#3498db')],
    labels=['Município líder', 'Demais municípios']
)
plt.tight_layout()
plt.show()

### 📝 Interpretação — Exercício 3

Os 10 municípios com mais notificações concentram grande parte dos casos do estado.
- **Vitória** e **Serra** (maiores cidades metropolitanas) tendem a liderar, o que é esperado pela densidade populacional;
- O destaque em vermelho facilita a identificação imediata do município líder;
- Esta concentração geográfica deve orientar políticas públicas de saúde.

---
## Exercício 4 — Distribuição por Sexo

**Objetivo:** Analisar a proporção de casos notificados entre os sexos.

In [ ]:
# Conta registros por sexo, ignorando valores nulos
sexo = df['Sexo'].value_counts(dropna=True)

# --- Gráfico de pizza ---
fig, ax = plt.subplots(figsize=(8, 8))

cores_pizza = ['#3498db', '#e91e63', '#95a5a6']
explode = [0.05] * len(sexo)  # Destaca levemente cada fatia

wedges, texts, autotexts = ax.pie(
    sexo.values,
    labels=sexo.index,
    autopct='%1.1f%%',
    colors=cores_pizza[:len(sexo)],
    explode=explode,
    startangle=140,
    textprops={'fontsize': 13}
)

# Deixar porcentagem em negrito
for autotext in autotexts:
    autotext.set_fontweight('bold')

ax.set_title('👥 Distribuição de Notificações por Sexo', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(sexo)

### 📝 Interpretação — Exercício 4

O gráfico de pizza mostra a divisão das notificações entre os sexos.
- Em geral, mulheres tendem a buscar mais atendimento/testagem, o que pode elevar sua proporção;
- A diferença entre os sexos pode refletir tanto exposição ao vírus quanto comportamento de busca por saúde;
- Casos com sexo não informado (Ignorado) devem ser levados em conta na análise.

---
## Exercício 5 — Casos por Faixa Etária

**Objetivo:** Entender quais faixas etárias concentram mais casos notificados.

In [ ]:
# Identifica a coluna de faixa etária (pode variar no dataset)
col_faixa = [c for c in df.columns if 'aixa' in c or 'taria' in c.lower()]
print('Colunas encontradas:', col_faixa)

# Use o nome correto da coluna abaixo:
coluna_faixa = col_faixa[0] if col_faixa else 'FaixaEtaria'

# Conta por faixa etária e ordena pelo índice (ordem natural)
faixa = df[coluna_faixa].value_counts().sort_index()

# --- Gráfico de barras ---
fig, ax = plt.subplots(figsize=(13, 6))

ax.bar(faixa.index.astype(str), faixa.values,
       color='#2980b9', edgecolor='white', width=0.7)

ax.set_title('👶🧑👴 Distribuição de Casos por Faixa Etária', fontsize=14, fontweight='bold')
ax.set_xlabel('Faixa Etária')
ax.set_ylabel('Número de Notificações')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 📝 Interpretação — Exercício 5

A análise por faixa etária mostra como a doença afetou diferentes grupos:
- A faixa de **30 a 59 anos** costuma concentrar mais notificações (população economicamente ativa, maior exposição);
- Idosos (60+), embora com menos notificações, têm maior risco de complicações;
- Essa informação é essencial para campanhas de vacinação e priorização de grupos.

---
## Exercício 6 — Taxa de Letalidade

**Objetivo:** Calcular a taxa de letalidade entre os casos confirmados de COVID-19.

In [ ]:
# Filtra apenas casos confirmados de COVID-19
confirmados = df[df['Classificacao'].str.contains('Confirmado', case=False, na=False)]

print(f'Total de casos confirmados: {len(confirmados):,}')

# Conta a evolução dos casos (Cura, Óbito, etc.)
evolucao = confirmados['Evolucao'].value_counts()
evolucao_pct = (evolucao / evolucao.sum() * 100).round(2)

tabela_evolucao = pd.DataFrame({'Quantidade': evolucao, 'Porcentagem (%)': evolucao_pct})
display(tabela_evolucao)

# Calcula a taxa de letalidade
obitos = confirmados['Evolucao'].str.contains('bito', case=False, na=False).sum()
taxa_letalidade = (obitos / len(confirmados) * 100).round(2)
print(f'\n💀 Óbitos confirmados: {obitos:,}')
print(f'📉 Taxa de Letalidade: {taxa_letalidade}%')

# --- Gráfico de barras da evolução ---
fig, ax = plt.subplots(figsize=(10, 5))
cores_evolucao = ['#2ecc71', '#e74c3c', '#f39c12', '#95a5a6']
ax.bar(evolucao.index, evolucao.values,
       color=cores_evolucao[:len(evolucao)], edgecolor='white')
ax.set_title(f'📉 Evolução dos Casos Confirmados (Letalidade: {taxa_letalidade}%)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Evolução')
ax.set_ylabel('Quantidade')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 📝 Interpretação — Exercício 6

A taxa de letalidade é um indicador crítico de saúde pública:
- Representa o percentual de óbitos entre os casos confirmados;
- Taxas acima de 2–3% indicavam situação preocupante nos picos da pandemia;
- A maioria dos confirmados evoluiu para **cura**, o que demonstra a capacidade do sistema de saúde.

---
## Exercício 7 — Sintomas Mais Frequentes

**Objetivo:** Identificar quais sintomas aparecem com mais frequência nas notificações.

In [ ]:
# Lista de colunas de sintomas (ajuste conforme o dataset real)
cols_sintomas = [c for c in df.columns if any(
    s in c for s in ['Febre','Tosse','Dispneia','Desconforto','Saturacao',
                     'Diarreia','Vomito','Dor','Olfato','Paladar','Fadiga', 'Cefaleia']
)]

print(f'Colunas de sintomas encontradas ({len(cols_sintomas)}): {cols_sintomas}')

# Conta quantos 'Sim' existem em cada coluna de sintoma
sintomas_sim = {}
for col in cols_sintomas:
    sintomas_sim[col] = (df[col].str.strip().str.lower() == 'sim').sum()

# Converte para Series e ordena
sintomas_serie = pd.Series(sintomas_sim).sort_values(ascending=True)

# --- Gráfico de barras horizontal ---
fig, ax = plt.subplots(figsize=(11, max(5, len(sintomas_serie) * 0.6)))

ax.barh(sintomas_serie.index, sintomas_serie.values,
        color='#16a085', edgecolor='white')

for i, (nome, valor) in enumerate(sintomas_serie.items()):
    ax.text(valor + sintomas_serie.max() * 0.01, i,
            f'{valor:,}', va='center', fontsize=10)

ax.set_title('🤒 Sintomas Mais Frequentes nas Notificações', fontsize=14, fontweight='bold')
ax.set_xlabel('Número de Registros com "Sim"')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 📝 Interpretação — Exercício 7

O perfil sintomático revela o quadro clínico predominante:
- **Tosse**, **Febre** e **Cefaleia** costumam ser os sintomas mais comuns da COVID-19;
- Sintomas como **perda de olfato/paladar** são marcadores específicos da doença;
- Esta análise auxilia na definição de critérios de triagem clínica.

---
## Exercício 8 — Comorbidades nos Óbitos por COVID-19

**Objetivo:** Verificar quais comorbidades estão mais presentes nos óbitos confirmados.

In [ ]:
# Filtra apenas óbitos por COVID confirmado
obitos_df = df[
    df['Classificacao'].str.contains('Confirmado', case=False, na=False) &
    df['Evolucao'].str.contains('bito', case=False, na=False)
]

print(f'Total de óbitos por COVID confirmado: {len(obitos_df):,}')

# Identifica colunas de comorbidades
cols_comorbidades = [c for c in df.columns if any(
    s in c for s in ['Diabetes','Cardio','Renal','Imuno','Obesi','Asma',
                     'Hepati','Neurolo','Pneumo','Pulmao','Hipertensao']
)]

print(f'\nComorbidades encontradas: {cols_comorbidades}')

# Conta 'Sim' por comorbidade nos óbitos
comorbidades_sim = {}
for col in cols_comorbidades:
    comorbidades_sim[col] = (obitos_df[col].str.strip().str.lower() == 'sim').sum()

comorb_serie = pd.Series(comorbidades_sim).sort_values(ascending=True)

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(11, max(5, len(comorb_serie) * 0.7)))

ax.barh(comorb_serie.index, comorb_serie.values,
        color='#c0392b', edgecolor='white')

for i, (nome, valor) in enumerate(comorb_serie.items()):
    ax.text(valor + comorb_serie.max() * 0.01, i,
            f'{valor:,}', va='center', fontsize=10)

ax.set_title('⚠️ Comorbidades Presentes nos Óbitos por COVID-19', fontsize=13, fontweight='bold')
ax.set_xlabel('Número de Óbitos com a Comorbidade')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

### 📝 Interpretação — Exercício 8

As comorbidades são fatores de risco amplamente documentados para agravamento da COVID-19:
- **Diabetes**, **Doenças Cardiovasculares** e **Hipertensão** aparecem com frequência nos óbitos;
- Esses dados reforçam a importância da priorização desses grupos nos protocolos de saúde;
- A ausência de comorbidades em parte dos óbitos indica que nenhum grupo estava isento de risco.

---
## Exercício 9 — Evolução Temporal dos Casos

**Objetivo:** Visualizar como as notificações evoluíram ao longo do tempo (por mês).

In [ ]:
# Identifica a coluna de data
cols_data = [c for c in df.columns if 'ata' in c.lower() and 'otifica' in c.lower()]
print('Colunas de data encontradas:', cols_data)

coluna_data = cols_data[0] if cols_data else 'DataNotificacao'

# Converte coluna para tipo datetime
# errors='coerce' transforma valores inválidos em NaT (nulo de data)
df['data_convertida'] = pd.to_datetime(df[coluna_data], errors='coerce', dayfirst=True)

# Cria coluna com Ano-Mês (ex: 2021-03)
df['ano_mes'] = df['data_convertida'].dt.to_period('M')

# Agrupa por mês e conta notificações
evolucao_temporal = df.groupby('ano_mes').size().sort_index()

# --- Gráfico de linha ---
fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(evolucao_temporal.index.astype(str),
        evolucao_temporal.values,
        color='#2980b9', linewidth=2.5, marker='o', markersize=5)

ax.fill_between(range(len(evolucao_temporal)),
                evolucao_temporal.values,
                alpha=0.2, color='#2980b9')

ax.set_title('📈 Evolução Mensal de Notificações de COVID-19 — ES', fontsize=14, fontweight='bold')
ax.set_xlabel('Mês')
ax.set_ylabel('Número de Notificações')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

### 📝 Interpretação — Exercício 9

O gráfico de linha temporal revela os padrões da pandemia no ES:
- Os **picos de notificação** correspondem às ondas epidemiológicas (variantes Delta, Ômicron, etc.);
- Períodos de queda indicam maiores restrições, vacinação ou sazonalidade;
- A análise temporal é uma das ferramentas mais poderosas para entender o comportamento de doenças infecciosas.

---
## Exercício 10 — Tabela Cruzada: Letalidade por Município

**Objetivo:** Cruzar município × evolução para calcular taxa de letalidade por região.

In [ ]:
# Filtra apenas casos confirmados
confirmados = df[df['Classificacao'].str.contains('Confirmado', case=False, na=False)].copy()

# Cria coluna binária: 1 = óbito, 0 = outros
confirmados['eh_obito'] = confirmados['Evolucao'].str.contains('bito', case=False, na=False).astype(int)

# Agrupa por município: total de casos e total de óbitos
letalidade_mun = confirmados.groupby('Municipio').agg(
    total_casos=('Municipio', 'count'),
    total_obitos=('eh_obito', 'sum')
).reset_index()

# Calcula a taxa de letalidade (%)
letalidade_mun['taxa_letalidade_pct'] = (
    letalidade_mun['total_obitos'] / letalidade_mun['total_casos'] * 100
).round(2)

# Filtra municípios com pelo menos 50 casos (para maior confiabilidade)
letalidade_mun = letalidade_mun[letalidade_mun['total_casos'] >= 50]

# Top 15 com maior taxa de letalidade
top_letal = letalidade_mun.sort_values('taxa_letalidade_pct', ascending=False).head(15)

print('📊 Top 15 Municípios por Taxa de Letalidade (mín. 50 casos):')
display(top_letal.reset_index(drop=True))

# --- Gráfico ---
fig, ax = plt.subplots(figsize=(11, 6))
ax.barh(top_letal['Municipio'][::-1],
        top_letal['taxa_letalidade_pct'][::-1],
        color='#e74c3c', edgecolor='white')

ax.set_title('💀 Taxa de Letalidade por Município (Top 15)', fontsize=13, fontweight='bold')
ax.set_xlabel('Taxa de Letalidade (%)')
ax.axvline(x=letalidade_mun['taxa_letalidade_pct'].mean(),
           color='navy', linestyle='--', label=f'Média ES: {letalidade_mun["taxa_letalidade_pct"].mean():.1f}%')
ax.legend()
plt.tight_layout()
plt.show()

### 📝 Interpretação — Exercício 10

A tabela cruzada é uma ferramenta poderosa para análise multidimensional:
- Municípios com alta taxa de letalidade podem indicar **subnotificação** (poucos casos, muitos óbitos) ou **infraestrutura hospitalar deficiente**;
- A linha tracejada da média estadual serve como referência comparativa;
- O filtro de mínimo 50 casos garante maior confiabilidade estatística dos resultados.

---
> 🎓 **KDD Concluído!** — Passamos pelas etapas de Seleção → Pré-processamento → Transformação → Mineração → Interpretação dos dados de COVID-19 no Espírito Santo.